# **0. Introduction:**

In [ ]:
import os
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import math
from pandas.plotting import scatter_matrix

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW

In [ ]:
try:
    from tqdm.notebook import tqdm
except ImportError:
    print("Warning: tqdm.notebook not available, falling back to regular tqdm")
    from tqdm import tqdm

# **1. EDA**

In [ ]:
import kagglehub
ROOT = kagglehub.dataset_download("zaber666/meld-dataset")
print("Đường dẫn ROOT hiện tại là:", ROOT)



In [ ]:
cpt = 0
for dirname, _, filenames in os.walk(ROOT):
    for filename in filenames:
        cpt += 1
        print(os.path.join(dirname, filename))
        if cpt > 5:
            break

In [ ]:
# Distribution graphs (histogram/bar graph) of column data
def plotPerColumnDistribution(df, nGraphShown, nGraphPerRow):
    nCol = df.shape[1]
    nGraphRow = int(math.ceil(nGraphShown / float(nGraphPerRow)))  # Ensure it's an int

    plt.figure(num=None, figsize=(6 * nGraphPerRow, 8 * nGraphRow), dpi=80, facecolor='w', edgecolor='k')

    for i in range(min(nCol, nGraphShown)):
        plt.subplot(nGraphRow, nGraphPerRow, i + 1)
        columnDf = df.iloc[:, i]

        if not np.issubdtype(columnDf.dtype, np.number):
            valueCounts = columnDf.value_counts()
            valueCounts.plot(kind='bar')
        else:
            columnDf.hist()
        
        plt.ylabel('counts')
        plt.title(f'{columnDf.name} (column {i})')

    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation matrix
def plotCorrelationMatrix(df, graphWidth):
    df = df.dropna(axis='columns')  # Drop columns with NaN
    df = df[[col for col in df if df[col].nunique() > 1]]  # Keep columns with more than one unique value

    if df.shape[1] < 2:
        print("Not enough columns for correlation matrix.")
        return

    corr = df.corr(numeric_only=True)

    plt.figure(figsize=(graphWidth, graphWidth))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True, cbar_kws={"shrink": .8})
    plt.title('Correlation Matrix')
    plt.show()

In [ ]:
# Scatter and density plots
def plotScatterMatrix(df, plotSize, textSize):
    df = df.select_dtypes(include=[np.number])  # Keep only numerical columns
    df = df.dropna(axis='columns')  # Drop columns with NaN
    df = df[[col for col in df if df[col].nunique() > 1]]  # Keep columns with >1 unique value

    if df.shape[1] > 10:
        print("Too many numerical columns for scatter matrix, reducing to first 10.")
        df = df.iloc[:, :10]

    scatter_matrix(df, alpha=0.75, figsize=(plotSize, plotSize), diagonal='hist')

    plt.suptitle('Scatter Matrix', fontsize=textSize)
    plt.show()

## 1.1. dev_sent_emo.csv

In [ ]:
nRowsRead = None # specify 'None' if want to read whole file
# dev_sent_emo.csv may have more rows in reality, but we are only loading/previewing the first 1000 rows
df1 = pd.read_csv(ROOT + '/MELD-RAW/MELD.Raw/dev_sent_emo.csv', delimiter=',', nrows = nRowsRead)
df1.dataframeName = 'dev_sent_emo.csv'
nRow, nCol = df1.shape
print(f'There are {nRow} rows and {nCol} columns')

In [ ]:
df1.head(10)

### Distribution graphs

In [ ]:
plotPerColumnDistribution(df1, 10, 5)

### Correlation matrix

In [ ]:
plotCorrelationMatrix(df1, 8)

### Scatter and Density plot

In [ ]:
plotScatterMatrix(df1, 15, 10)

## 1.2. test_sent_emo.csv

In [ ]:
nRowsRead = None # specify 'None' if want to read whole file
# test_sent_emo.csv may have more rows in reality, but we are only loading/previewing the first 1000 rows
df2 = pd.read_csv(ROOT + '/MELD-RAW/MELD.Raw/test_sent_emo.csv', delimiter=',', nrows = nRowsRead)
df2.dataframeName = 'test_sent_emo.csv'
nRow, nCol = df2.shape
print(f'There are {nRow} rows and {nCol} columns')

In [ ]:
df2.head(10)

### Distribution graphs

In [ ]:
plotPerColumnDistribution(df2, 10, 5)

### Correlation matrix

In [ ]:
plotCorrelationMatrix(df2, 8)

### Scatter and Density plot

In [ ]:
plotScatterMatrix(df2, 15, 10)

## 1.3. train_sent_emo.csv

In [ ]:
nRowsRead = None # specify 'None' if want to read whole file
# train_sent_emo.csv may have more rows in reality, but we are only loading/previewing the first 1000 rows
df3 = pd.read_csv(ROOT + '/MELD-RAW/MELD.Raw/train/train_sent_emo.csv', delimiter=',', nrows = nRowsRead)
df3.dataframeName = 'train_sent_emo.csv'
nRow, nCol = df3.shape
print(f'There are {nRow} rows and {nCol} columns')

In [ ]:
df3.head(10)

### Distribution graphs

In [ ]:
plotPerColumnDistribution(df3, 10, 5)

### Correlation matrix

In [ ]:
plotCorrelationMatrix(df3, 8)

### Scatter and Density plots

In [ ]:
plotScatterMatrix(df3, 15, 10)

## 1.4. Khám phá dữ liệu chuyên sâu cho NLP & Lưu biểu đồ cho Streamlit


In [ ]:


# Đọc lại dữ liệu Train và lọc bỏ các dòng bị thiếu nhãn (để vẽ cho chính xác)
train_df_eda = pd.read_csv(ROOT + '/MELD-RAW/MELD.Raw/train/train_sent_emo.csv')
train_df_eda = train_df_eda.dropna(subset=['Emotion'])

print(f"\nSố lượng dữ liệu tập Train dùng để vẽ: {train_df_eda.shape[0]} câu thoại.")

# Vẽ và lưu biểu đồ Phân bố Nhãn (Class Distribution)
plt.figure(figsize=(10, 5))
sns.countplot(data=train_df_eda, x='Emotion', order=train_df_eda['Emotion'].value_counts().index, palette='viridis')
plt.title("Phân bố số lượng các nhãn cảm xúc trong tập Train")
plt.xlabel("Cảm xúc")
plt.ylabel("Số lượng câu")
plt.savefig("class_distribution.png", bbox_inches='tight') # <-- LƯU ẢNH LẠI
plt.show()

# Vẽ và lưu Histogram Chiều dài câu văn (Sentence Length)
train_df_eda['word_count'] = train_df_eda['Utterance'].apply(lambda x: len(str(x).split()))
plt.figure(figsize=(10, 5))
sns.histplot(train_df_eda['word_count'], bins=50, kde=True, color='teal')
plt.title("Histogram: Phân bố số lượng từ trong câu thoại")
plt.xlabel("Số lượng từ")
plt.ylabel("Tần suất")
plt.savefig("sentence_length_histogram.png", bbox_inches='tight') # <-- LƯU ẢNH LẠI
plt.show()

print("Đã lưu thành công 2 ảnh: class_distribution.png và sentence_length_histogram.png để dùng cho Streamlit.")

# **2. Implementation of Multimodal EmotionLines Dataset (MELD):**

In [ ]:
!git clone https://github.com/declare-lab/MELD.git

# **3. Loading MELD Dataset**

In [ ]:
train_df = pd.read_csv(ROOT + '/MELD-RAW/MELD.Raw/train/train_sent_emo.csv')
val_df = pd.read_csv(ROOT + '/MELD-RAW/MELD.Raw/dev_sent_emo.csv')
test_df = pd.read_csv(ROOT + '/MELD-RAW/MELD.Raw/test_sent_emo.csv')

In [ ]:
# Filter out rows with missing emotion labels
train_df = train_df.dropna(subset=['Emotion'])
val_df = val_df.dropna(subset=['Emotion'])
test_df = test_df.dropna(subset=['Emotion'])

# **4. Encode Emotion Labels**

In [ ]:
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['Emotion'])
val_df['label'] = le.transform(val_df['Emotion'])
test_df['label'] = le.transform(test_df['Emotion'])

In [ ]:
label_map = dict(zip(le.classes_, le.transform(le.classes_)))

# **5. Tokenization:**

In [ ]:
from transformers import AutoTokenizer


# LẦN 1: Để nguyên dòng dưới đây để train BERT
MODEL_NAME = 'bert-base-uncased' 

# LẦN 2: Khi muốn train DistilRoBERTa, hãy thêm dấu # vào đầu dòng trên, và xóa dấu # ở dòng dưới:
# MODEL_NAME = 'distilroberta-base' 

# Tự động tạo tên thư mục lưu model tương ứng (để dùng cho Bước 10)
SAVE_DIR = f"./{MODEL_NAME.split('/')[-1]}-emotion" 

# Dùng AutoTokenizer thay vì BertTokenizer cứng nhắc
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts, labels, max_len=128):
    encodings = tokenizer(list(texts), padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    return encodings, labels

# **6. PyTorch Dataset and Dataloader**

In [ ]:
class MELDDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        return {
            # ĐÃ XÓA torch.tensor() bọc bên ngoài vì tokenizer đã trả về tensor rồi
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

    def __len__(self):
        return len(self.labels)

In [ ]:
train_enc, train_labels = tokenize(train_df['Utterance'], train_df['label'])
val_enc, val_labels = tokenize(val_df['Utterance'], val_df['label'])
test_enc, test_labels = tokenize(test_df['Utterance'], test_df['label'])

In [ ]:
train_dataset = MELDDataset(train_enc, train_labels)
val_dataset = MELDDataset(val_enc, val_labels)
test_dataset = MELDDataset(test_enc, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

# **7. Defining Model**

In [ ]:
from transformers import AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Dùng AutoModel để nó tự động nhận diện kiến trúc dựa vào biến MODEL_NAME ở Mục 5
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(le.classes_))
model.to(device)

# **8. Training**

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup
import torch.nn as nn
import numpy as np

# 1. TÍNH TOÁN TRỌNG SỐ (CLASS WEIGHTS) ĐỂ TRỊ MẤT CÂN BẰNG DỮ LIỆU
# Lấy toàn bộ nhãn từ tập train để tính toán
y_train = [train_dataset[i]['labels'].item() for i in range(len(train_dataset))]
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Định nghĩa hàm Loss có gắn trọng số (phạt nặng nếu đoán sai các nhãn thiểu số như fear, disgust)
loss_fn = nn.CrossEntropyLoss(weight=weights)

# 2. CẤU HÌNH THÔNG SỐ TRAINING
epochs = 4 # Giảm từ 100 xuống 4 để mô hình không bị "học vẹt" (Overfitting)
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

# Thêm Scheduler để điều chỉnh tốc độ học (Learning Rate) mượt mà hơn
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

train_losses = []
val_accuracies = []

def train(model, loader):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc="Training"):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Chạy model, lấy kết quả thô (logits)
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Tự tính sai số (loss) bằng hàm loss_fn có trọng số đã khai báo ở trên
        loss = loss_fn(logits, labels)

        total_loss += loss.item()
        loss.backward()
        
        # Cắt xén gradient để tránh lỗi bùng nổ toán học
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step() # Cập nhật tốc độ học

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    accuracy = correct / total
    return total_loss / len(loader), accuracy

def evaluate(model, loader):
    model.eval()
    preds, true_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(true_labels, preds)
    report = classification_report(true_labels, preds, target_names=le.classes_, output_dict=False)
    return report, accuracy

# 3. VÒNG LẶP HUẤN LUYỆN VÀ LƯU MODEL TỐT NHẤT
best_val_acc = 0 # Biến để theo dõi model xịn nhất

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    train_loss, train_acc = train(model, train_loader)
    train_losses.append(train_loss)

    print(f"Training Loss: {train_loss:.4f} | Training Accuracy: {train_acc:.4f}")

    report, val_acc = evaluate(model, val_loader)
    val_accuracies.append(val_acc)

    print(f"Validation Accuracy: {val_acc:.4f}")
    print("Validation Report:\n", report)
    
    # CƠ CHẾ LƯU TỰ ĐỘNG: Nếu epoch này thi được điểm cao hơn epoch trước, thì lưu model lại!
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print(f"Đạt mốc Validation Accuracy mới: {best_val_acc:.4f}. Đang lưu model...")
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)

# **9. Confusion Matrix & Training History**


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. TẢI LẠI MODEL TỐT NHẤT vì trong lúc train, có thể epoch 3 tốt hơn epoch 4, 
print(f"Đang tải lại phiên bản model tốt nhất từ thư mục: {SAVE_DIR} ...")
best_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(device)

print("\n--- ĐÁNH GIÁ TRÊN TẬP TEST ---")
test_report, test_acc = evaluate(best_model, test_loader)
print(f"Test Accuracy: {test_acc:.4f}")
print("Test Report:\n", test_report)

# 2. VẼ VÀ LƯU CONFUSION MATRIX (Ma trận nhầm lẫn)
best_model.eval()
preds, true_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()

        logits = best_model(input_ids, attention_mask=attention_mask).logits
        pred = torch.argmax(logits, dim=1).cpu().numpy()

        preds.extend(pred)
        true_labels.extend(labels)

cm = confusion_matrix(true_labels, preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Dự đoán (Predicted)")
plt.ylabel("Thực tế (True)")
plt.title(f"Confusion Matrix (Test Set) - {MODEL_NAME}")
# LƯU ẢNH LẠI ĐỂ MỐT ĐƯA LÊN STREAMLIT
plt.savefig(f"confusion_matrix_{MODEL_NAME.split('/')[-1]}.png", bbox_inches='tight') 
plt.show()

# 3. VẼ VÀ LƯU BIỂU ĐỒ TRAINING HISTORY (Loss & Accuracy)
epochs_range = range(1, epochs + 1)
plt.figure(figsize=(12, 5))

# Biểu đồ Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, label='Training Loss', marker='o')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Over Epochs")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

# Biểu đồ Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs_range, val_accuracies, label='Validation Accuracy', color='green', marker='o')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Over Epochs")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
# LƯU ẢNH LẠI ĐỂ MỐT ĐƯA LÊN STREAMLIT
plt.savefig(f"training_history_{MODEL_NAME.split('/')[-1]}.png", bbox_inches='tight')
plt.show()

print(f"✅ Đã lưu ảnh Confusion Matrix và biểu đồ Training của {MODEL_NAME} thành công!")



# **10. Inference**


In [ ]:
print("\n--- THỬ NGHIỆM DỰ ĐOÁN CÂU THỰC TẾ ---")
inf_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)

def predict_emotion(text, max_len=128):
    enc = inf_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=max_len)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = best_model(**enc).logits
        pred_id = int(torch.argmax(logits, dim=1).cpu().item())
    return le.inverse_transform([pred_id])[0]

# Đưa một vài câu test thử xem model có khôn không
print("Câu 1: I can't believe this happened!!! -> Dự đoán:", predict_emotion("I can't believe this happened!!!"))
print("Câu 2: That's awesome, I'm so happy today. -> Dự đoán:", predict_emotion("That's awesome, I'm so happy today."))
print("Câu 3: Get out of my face right now! -> Dự đoán:", predict_emotion("Get out of my face right now!"))